# Downstream Retrieval And PCST

This notebook uses the main project config to load its downstream database.
In practice that means it reads the configured node table, edge table, and final GCN embeddings.

It is split into two modules:
1. Retrieve a full, unpruned subgraph.
2. Apply PCST to prune that subgraph.

In [29]:
from pathlib import Path

import networkx as nx
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import yaml
from pcst_fast import pcst_fast
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "configs").exists() and (REPO_ROOT / ".." / "configs").exists():
    REPO_ROOT = (REPO_ROOT / "..").resolve()

def resolve_path(path_str: str) -> Path:
    path = Path(path_str)
    if path.is_absolute():
        return path
    return REPO_ROOT / path

with open(resolve_path("configs/config.yaml"), "r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

# Load the downstream database from the main project config.
nodes = pd.read_csv(resolve_path(config["paths"]["nodes_csv"]))
edges = pd.read_csv(resolve_path(config["paths"]["edges_csv"]))
node_embeddings = np.load(resolve_path(config["paths"]["final_embeddings"]))

# Keep node ids as strings and align the embedding matrix with nodes.csv row order.
nodes = nodes.copy()
nodes["node_id"] = nodes["node_id"].astype(str)
edges = edges.copy()
edges["source"] = edges["source"].astype(str)
edges["target"] = edges["target"].astype(str)

# Build a direct lookup from node id to its embedding.
node_ids = nodes["node_id"].tolist()
node_emb_lookup = dict(zip(node_ids, node_embeddings))

embedding_model_name = config["embedding"]["model_name"]
# Query embeddings are generated from the same model family configured for node features.
node_texts = nodes["text"].fillna("").astype(str)
node_texts = node_texts.where(node_texts.ne(""), nodes["title"].fillna("").astype(str))
query_encoder = SentenceTransformer(embedding_model_name)
embedding_dim = int(node_embeddings.shape[1])


/opt/anaconda3/envs/ml_env/lib/python3.11/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


## Shared Helpers

In [30]:
def build_graph(edges_df, node_ids=None):
    """Create a simple undirected NetworkX graph from edge rows.

    Parameters
    ----------
    edges_df:
        DataFrame with `source` and `target` columns.
    node_ids:
        Optional node ids to add even if they are isolated.
    """
    graph = nx.Graph()

    if node_ids is not None:
        for node_id in node_ids:
            graph.add_node(node_id)

    for row in edges_df.itertuples(index=False):
        if row.source != row.target:
            graph.add_edge(row.source, row.target)

    return graph

def get_text_rows(node_ids, nodes_df=nodes):
    """Return article / paragraph rows in the same order as the input ids."""
    requested = [str(node_id) for node_id in node_ids]
    order = {node_id: idx for idx, node_id in enumerate(requested)}

    matches = nodes_df[nodes_df["node_id"].isin(requested)].copy()
    matches["request_order"] = matches["node_id"].map(order)

    return matches.sort_values("request_order")[["node_id", "title", "type", "text"]].reset_index(drop=True)

def plot_subgraph(graph, title, nodes_df=nodes, score_lookup=None):
    """Visualise any retrieved graph with Plotly.

    If `score_lookup` is provided, similarity / prize values are shown in the hover text
    and the node size is increased by the prize value.
    """
    type_lookup = nodes_df[["node_id", "type"]].drop_duplicates("node_id").set_index("node_id")["type"].astype(str).to_dict()

    try:
        layout = nx.spring_layout(graph, seed=7)
    except TypeError:
        layout = nx.spring_layout(graph)

    edge_x = []
    edge_y = []
    for source, target in graph.edges():
        x0, y0 = layout[source]
        x1, y1 = layout[target]
        edge_x.extend([x0, x1, None])
        edge_y.extend([y0, y1, None])

    edge_trace = go.Scatter(
        x=edge_x,
        y=edge_y,
        mode="lines",
        line={"width": 1, "color": "#94a3b8"},
        hoverinfo="none",
    )

    palette = {
        "article": "#1f77b4",
        "paragraph": "#ff7f0e",
        "annex": "#2ca02c",
        "recital": "#8c564b",
    }

    node_x = []
    node_y = []
    node_text = []
    node_label = []
    node_color = []
    node_size = []

    for node_id in graph.nodes():
        x, y = layout[node_id]
        node_type = type_lookup.get(node_id, "unknown")
        score = {} if score_lookup is None else score_lookup.get(node_id, {})
        similarity = score.get("similarity", np.nan)
        prize = score.get("prize", 0.0)

        node_x.append(x)
        node_y.append(y)
        node_label.append(node_id)
        node_color.append(palette.get(node_type.lower(), "#7f7f7f"))
        node_size.append(16 + (6 * prize))
        node_text.append(
            f"node_id={node_id}<br>type={node_type}<br>degree={graph.degree(node_id)}"
            f"<br>similarity={similarity:.4f}<br>prize={prize:.2f}"
        )

    node_trace = go.Scatter(
        x=node_x,
        y=node_y,
        mode="markers+text",
        text=node_label,
        textposition="top center",
        hoverinfo="text",
        hovertext=node_text,
        marker={
            "size": node_size,
            "color": node_color,
            "line": {"width": 1, "color": "#ffffff"},
            "opacity": 0.9,
        },
    )

    fig = go.Figure(data=[edge_trace, node_trace])
    fig.update_layout(
        title=title,
        template="plotly_white",
        showlegend=False,
        xaxis={"showgrid": False, "zeroline": False, "showticklabels": False},
        yaxis={"showgrid": False, "zeroline": False, "showticklabels": False},
        margin={"l": 20, "r": 20, "t": 60, "b": 20},
        height=650,
    )

    return fig

# Build the full graph once. Both modules reuse it.
full_graph = build_graph(edges, node_ids=node_ids)

## Module 1: Retrieve An Unpruned Subgraph

In [43]:
def encode_query(query):
    """Encode the raw query with the embedding model from config.

    The returned query vector is used for both retrieval and PCST scoring.
    Its width must match the saved node embedding width.
    """
    query_emb = query_encoder.encode([query], normalize_embeddings=True, show_progress_bar=False)[0]
    if query_emb.shape[0] != embedding_dim:
        raise ValueError(
            f"Query embedding dimension {query_emb.shape[0]} does not match node embedding dimension {embedding_dim}."
        )
    return query_emb

def retrieve_seed_nodes(query_emb, node_embeds=node_embeddings, node_ids=node_ids, k=2):
    """Rank all nodes by cosine similarity to the query embedding and return the top-k node ids.

    Cosine similarity is computed in one embedding space:
    - query embedding: config['embedding']['model_name']
    - node embeddings: config['paths']['final_embeddings']
    """
    similarities = cosine_similarity([query_emb], node_embeds)[0]
    ranked_idx = np.argsort(similarities)[::-1][:min(k, len(similarities))]
    return [node_ids[idx] for idx in ranked_idx]

def expand_k_hops(graph, seeds, k=2):
    """Return the full k-hop subgraph around the seed nodes."""
    valid_seeds = [seed for seed in seeds if seed in graph]
    nodes_in_scope = set(valid_seeds)
    frontier = set(valid_seeds)

    for _ in range(k):
        next_frontier = set()
        for node_id in frontier:
            next_frontier.update(graph.neighbors(node_id))
        frontier = next_frontier
        nodes_in_scope.update(next_frontier)

    return graph.subgraph(nodes_in_scope).copy()

def retrieve_unpruned_subgraph(query, num_retrieved_seeds=5, k_hops=2):
    """Module 1.

    Input:
    - raw text query
    - number of cosine-similarity seed nodes
    - number of hop expansions

    Output:
    - the full retrieved graph before pruning
    - the corresponding article / paragraph rows
    - a Plotly visualisation
    """
    query_emb = encode_query(query)
    seed_nodes = retrieve_seed_nodes(query_emb, k=num_retrieved_seeds)
    unpruned_graph = expand_k_hops(full_graph, seed_nodes, k=k_hops)

    retrieved_node_ids = list(unpruned_graph.nodes())
    retrieved_edges_df = pd.DataFrame(list(unpruned_graph.edges()), columns=["source", "target"])
    retrieved_text_df = get_text_rows(retrieved_node_ids)
    retrieved_figure = plot_subgraph(
        unpruned_graph,
        title=f"Unpruned retrieved subgraph for query: {query}",
    )

    return {
        "query": query,
        "query_emb": query_emb,
        "seed_nodes": seed_nodes,
        "subgraph": unpruned_graph,
        "subgraph_node_ids": retrieved_node_ids,
        "subgraph_edges_df": retrieved_edges_df,
        "text_df": retrieved_text_df,
        "figure": retrieved_figure,
    }

## Module 2: Apply PCST To The Retrieved Subgraph

In [32]:
def assign_rank_prizes(node_ids, query_emb, node_emb_lookup, top_k=10):
    """Assign node prizes from cosine similarity ranks.

    Only the top-k most similar nodes receive non-zero prizes.
    Higher similarity means a larger prize.
    """
    embs = np.vstack([node_emb_lookup[node_id] for node_id in node_ids])
    similarities = cosine_similarity([query_emb], embs)[0]

    prizes = np.zeros(len(node_ids), dtype=float)
    ranked_idx = np.argsort(similarities)[::-1][:min(top_k, len(similarities))]
    for rank, idx in enumerate(ranked_idx):
        prizes[idx] = top_k - rank

    return prizes, similarities

def graph_to_pcst_input(graph, prizes, edge_cost=1.0):
    """Convert a NetworkX graph into the array representation required by pcst_fast."""
    node_list = list(graph.nodes())
    node_to_idx = {node_id: idx for idx, node_id in enumerate(node_list)}

    edge_pairs = []
    edge_costs = []
    for source, target in graph.edges():
        edge_pairs.append((node_to_idx[source], node_to_idx[target]))
        edge_costs.append(edge_cost)

    return (
        node_list,
        np.asarray(edge_pairs, dtype=np.int32),
        np.asarray(prizes, dtype=np.float64),
        np.asarray(edge_costs, dtype=np.float64),
    )

def run_node_prize_pcst(graph, prizes, edge_cost=1.0, pruning="strong"):
    """Run PCST and map the selected vertex / edge indices back to node ids."""
    node_list, edge_array, prize_array, cost_array = graph_to_pcst_input(graph, prizes, edge_cost=edge_cost)
    selected_vertex_idx, selected_edge_idx = pcst_fast(edge_array, prize_array, cost_array, -1, 1, pruning, 0)

    selected_nodes = [node_list[idx] for idx in selected_vertex_idx]
    selected_node_set = set(selected_nodes)
    selected_edges = []
    edge_list = list(graph.edges())
    for edge_idx in selected_edge_idx:
        source, target = edge_list[edge_idx]
        if source in selected_node_set and target in selected_node_set:
            selected_edges.append((source, target))

    return selected_nodes, selected_edges

def prune_retrieved_subgraph(retrieval_result, prize_top_k=10, edge_cost=1.0):
    """Module 2.

    Input:
    - the output of Module 1
    - PCST prize top-k setting
    - constant edge cost

    Output:
    - the pruned graph
    - the corresponding article / paragraph rows
    - a Plotly visualisation
    """
    candidate_graph = retrieval_result["subgraph"]
    candidate_node_ids = retrieval_result["subgraph_node_ids"]

    candidate_node_emb_lookup = {
        node_id: node_emb_lookup[node_id]
        for node_id in candidate_node_ids
    }

    prizes, similarities = assign_rank_prizes(
        node_ids=candidate_node_ids,
        query_emb=retrieval_result["query_emb"],
        node_emb_lookup=candidate_node_emb_lookup,
        top_k=prize_top_k,
    )

    selected_nodes, selected_edges = run_node_prize_pcst(
        candidate_graph,
        prizes,
        edge_cost=edge_cost,
        pruning="strong",
    )

    node_scores_df = pd.DataFrame({
        "node_id": candidate_node_ids,
        "similarity": similarities,
        "prize": prizes,
    }).sort_values(["prize", "similarity"], ascending=False)

    selected_edges_df = pd.DataFrame(selected_edges, columns=["source", "target"])
    pruned_graph = build_graph(selected_edges_df, node_ids=selected_nodes)
    pruned_text_df = get_text_rows(selected_nodes)
    score_lookup = node_scores_df.set_index("node_id").to_dict("index")
    pruned_figure = plot_subgraph(
        pruned_graph,
        title=f"PCST-pruned subgraph for query: {retrieval_result['query']}",
        score_lookup=score_lookup,
    )

    return {
        "query": retrieval_result["query"],
        "selected_nodes": selected_nodes,
        "selected_edges_df": selected_edges_df,
        "node_scores_df": node_scores_df,
        "pruned_graph": pruned_graph,
        "text_df": pruned_text_df,
        "figure": pruned_figure,
    }

## Example

In [46]:
retrieval_result = retrieve_unpruned_subgraph(
    query="What differentiates AI Systems and AI models?",
    num_retrieved_seeds=2,
    k_hops=1,
)

retrieval_result["text_df"]

,node_id,title,type,text
0,recital_175,Recital (175),recital,(175) In order to ensure uniform conditions fo...
1,definition_3,Definition: provider,definition,(3) ‘provider’ means a natural or legal person...
2,definition_64,Definition: high-impact capabilities,definition,(64) ‘high-impact capabilities’ means capabili...
3,definition_63,Definition: general-purpose AI model,definition,(63) ‘general-purpose AI model’ means an AI mo...
4,definition_2,Definition: risk,definition,(2) ‘risk’ means the combination of the probab...
5,definition_29,Definition: training data,definition,(29) ‘training data’ means data used for train...
6,recital_111,Recital (111),recital,(111) It is appropriate to establish a methodo...
7,definition_47,Definition: AI Office,definition,(47) ‘AI Office’ means the Commission’s functi...
8,definition_65,Definition: systemic risk,definition,(65) ‘systemic risk’ means a risk that is spec...
9,definition_9,Definition: placing on the market,definition,(9) ‘placing on the market’ means the first ma...


In [47]:
retrieval_result["figure"]

In [48]:
pruned_result = prune_retrieved_subgraph(
    retrieval_result,
    prize_top_k=5,
    edge_cost=1.0,
)

pruned_result["text_df"]

,node_id,title,type,text
0,definition_64,Definition: high-impact capabilities,definition,(64) ‘high-impact capabilities’ means capabili...
1,definition_29,Definition: training data,definition,(29) ‘training data’ means data used for train...
2,recital_111,Recital (111),recital,(111) It is appropriate to establish a methodo...


In [49]:
pruned_result["figure"]